# Week 2b: UNI Feature Extraction

Extract UNI foundation model features (1024-dim) to replace ResNet-50.

**Note:** UNI is a ViT-L model trained on 100M+ H&E pathology slides.
This is a significant upgrade from ImageNet-pretrained ResNet-50.


## Cell 1 — Setup and device

In [1]:
import torch
import pickle
from pathlib import Path
from tqdm import tqdm
import sys
import os

# VRAM management for RTX 5060 (8GB)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

sys.path.insert(0, '..')

DEVICE        = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

PATCHES_DIR   = Path('./data/patches')
FEATURES_DIR  = Path('./data/features_uni')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

print(f'\nPatches dir: {PATCHES_DIR}')
print(f'Patches found: {len(list(PATCHES_DIR.glob("*.pkl")))}')
print(f'Features output dir: {FEATURES_DIR}')

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA: 12.8

Patches dir: data/patches
Patches found: 333
Features output dir: data/features_uni


## Cell 2 — Load UNI model

In [4]:
from pathq.uni_extractor import build_uni_extractor

print('Loading UNI from HuggingFace...')
print('NOTE: Requires huggingface_hub login and UNI term acceptance')
print('      See: https://huggingface.co/MahmoodLab/uni')
print()

uni_model, uni_transform = build_uni_extractor(DEVICE)
print('✅ UNI loaded')

Loading UNI from HuggingFace...
NOTE: Requires huggingface_hub login and UNI term acceptance
      See: https://huggingface.co/MahmoodLab/uni

Loading UNI feature extractor from HuggingFace...
(First run downloads ~1.8GB — subsequent runs use cache)
UNI loaded: 303,350,784 params (all frozen)
Output dimension: 1024
✅ UNI loaded


In [6]:
# VRAM: UNI is ViT-L (bigger than ResNet-50), use smaller batch_size
BATCH_SIZE = 32  # Safe for RTX 5060 8GB (vs 128 for ResNet-50)

print(f'Extracting UNI features (batch_size={BATCH_SIZE})...')
print(f'UNI output: 1024-dim features per patch')
print()

for pkl_path in tqdm(list(PATCHES_DIR.glob('*.pkl')), desc='Extracting UNI features'):
    slide_id  = pkl_path.stem
    save_path = FEATURES_DIR / f'{slide_id}_uni_features.pt'

    if save_path.exists():
        continue  # Skip already processed

    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    patches = data['patches']   # list of PIL Images
    coords  = data['coords']    # list of (col, row) tuples

    all_feats = []
    with torch.no_grad():
        for i in range(0, len(patches), BATCH_SIZE):
            batch = torch.stack(
                [uni_transform(p) for p in patches[i:i+BATCH_SIZE]]
            ).to(DEVICE)
            feats = uni_model(batch)      # (B, 1024)
            all_feats.append(feats.cpu())

    features = torch.cat(all_feats, dim=0)              # (N, 1024)
    coords_t = torch.tensor(coords, dtype=torch.float32) # (N, 2)

    torch.save({
        'slide_id':  slide_id,
        'features':  features,    # (N, 1024)
        'coords':    coords_t,    # (N, 2)
        'n_patches': len(patches),
    }, save_path)

print('✅ Done. Feature files saved to:', FEATURES_DIR)
print(f'Total files: {len(list(FEATURES_DIR.glob("*.pt")))}')

Extracting UNI features (batch_size=32)...
UNI output: 1024-dim features per patch



Extracting UNI features: 100%|██████████| 333/333 [42:23<00:00,  7.64s/it]

✅ Done. Feature files saved to: data/features_uni
Total files: 333


## Cell 3 — Extract and save UNI features

In [ ]:
# VRAM: UNI is ViT-L (bigger than ResNet-50), use smaller batch_size
BATCH_SIZE = 32  # Safe for RTX 5060 8GB (vs 128 for ResNet-50)

print(f'Extracting UNI features (batch_size={BATCH_SIZE})...')
print(f'UNI output: 1024-dim features per patch')
print()

for pkl_path in tqdm(list(PATCHES_DIR.glob('*.pkl')), desc='Extracting UNI features'):
    slide_id  = pkl_path.stem
    save_path = FEATURES_DIR / f'{slide_id}_uni_features.pt'

    if save_path.exists():
        continue  # Skip already processed

    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    patches = data['patches']   # list of PIL Images
    coords  = data['coords']    # list of (col, row) tuples

    all_feats = []
    with torch.no_grad():
        for i in range(0, len(patches), BATCH_SIZE):
            batch = torch.stack(
                [uni_transform(p) for p in patches[i:i+BATCH_SIZE]]
            ).to(DEVICE)
            feats = uni_model(batch)      # (B, 1024)
            all_feats.append(feats.cpu())

    features = torch.cat(all_feats, dim=0)              # (N, 1024)
    coords_t = torch.tensor(coords, dtype=torch.float32) # (N, 2)

    torch.save({
        'slide_id':  slide_id,
        'features':  features,    # (N, 1024)
        'coords':    coords_t,    # (N, 2)
        'n_patches': len(patches),
    }, save_path)

print('✅ Done. Feature files saved to:', FEATURES_DIR)
print(f'Total files: {len(list(FEATURES_DIR.glob("*.pt")))}')

## Cell 4 — Verify extraction

In [7]:
import numpy as np

# Check one file
fp = list(FEATURES_DIR.glob('*.pt'))[0]
d  = torch.load(fp, weights_only=False)

print(f'Slide: {d["slide_id"]}')
print(f'Features shape: {d["features"].shape}')   # expect (N, 1024)
print(f'Coords shape:   {d["coords"].shape}')     # expect (N, 2)
print(f'Feature range:  [{d["features"].min():.3f}, {d["features"].max():.3f}]')
print(f'Feature mean:   {d["features"].mean():.3f}')
print(f'Feature std:    {d["features"].std():.3f}')
print()
print('✅ UNI extraction verified')

# Clean up
del uni_model
torch.cuda.empty_cache()
print('UNI model freed from GPU memory')

Slide: test_128
Features shape: torch.Size([3000, 1024])
Coords shape:   torch.Size([3000, 2])
Feature range:  [-6.932, 7.658]
Feature mean:   -0.003
Feature std:    1.222

✅ UNI extraction verified
UNI model freed from GPU memory


In [ ]:
import torch
x = torch.randn(2, 3, 224, 224).cuda()
print("✅ GPU works")

In [2]:
import torch
import timm
import pennylane as qml
from torch_geometric.nn import GATConv
import torch.nn as nn

device = torch.device('cuda')
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
print(f'GPU:      {torch.cuda.get_device_name(0)}')
print(f'Compute:  {torch.cuda.get_device_capability(0)}')

# UNI
model = timm.create_model(
    'hf-hub:MahmoodLab/uni',
    pretrained=True, init_values=1e-5, dynamic_img_size=True
).to(device).eval()
x = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(x)
print(f'UNI:      {out.shape}  on {out.device}')

# GATConv
gat = GATConv(256, 256, heads=4, concat=False).to(device)
print(f'GATConv:  OK')

# GRU fallback
gru = nn.GRU(256, 128, batch_first=True, bidirectional=True).to(device)
t = torch.randn(1, 50, 256).to(device)
out2, _ = gru(t)
print(f'GRU:      {out2.shape}  on {out2.device}')

# VQC
dev = qml.device('lightning.qubit', wires=3)
print(f'PennyLane: lightning.qubit OK')

print()
print('All components working. Ready to train.')
print('Mamba will be installed on RunPod in Week 7.')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch:  2.11.0+cu128
CUDA:     True
GPU:      NVIDIA GeForce RTX 5060 Laptop GPU
Compute:  (12, 0)
UNI:      torch.Size([2, 1024])  on cuda:0
GATConv:  OK
GRU:      torch.Size([1, 50, 256])  on cuda:0
PennyLane: lightning.qubit OK

All components working. Ready to train.
Mamba will be installed on RunPod in Week 7.
